# Copula Theory in Statistical Inference and Financial Applications

A copula is a mathematical object that separates marginal distributions from the dependence structure of a multivariate random vector. In statistical inference, it provides a coherent framework to construct joint distributions from univariate marginals and a multivariate dependence function.

#### Theoretical foundations
Given a random vector $X = (X_1, \ldots, X_d)$ with joint cumulative distribution function $F$ and marginal CDFs $F_i$, Sklar's theorem states that there exists a copula $C$ such that

$$ F(x_1, \ldots, x_d) = C\big(F_1(x_1), \ldots, F_d(x_d)\big). $$

When the marginals are continuous, the copula $C$ is unique. If densities exist, the joint density admits the decomposition

$$ f(x_1, \ldots, x_d) = c\big(F_1(x_1), \ldots, F_d(x_d)\big) \prod_{i=1}^d f_i(x_i), $$

where

$$ c(u_1, \ldots, u_d) = \frac{\partial^d}{\partial u_1 \cdots \partial u_d}C(u_1, \ldots, u_d). $$

This decomposition highlights two inferential components:
- Marginal inference: estimation of each $F_i$ or $f_i$.
- Dependence inference: estimation of the copula $C$ or its density $c$.

#### Inference methodology
Copula inference is typically organized in one of the following approaches:

- Parametric inference:
    - Specify parametric marginals $F_i(x_i; \theta_i)$ and a parametric copula $C(u_1, \ldots, u_d; \psi)$.
    - Estimate parameters by full maximum likelihood:

        $$ \hat{\theta}, \hat{\psi} = \arg\max_{\theta, \psi} \sum_{t=1}^n \log f(x_{t1}, \ldots, x_{td}; \theta, \psi). $$

- Inference Functions for Margins (IFM):
    - Step 1: estimate marginal parameters $\hat{\theta}_i$ independently.
    - Step 2: estimate copula parameter $\hat{\psi}$ using pseudo-observations $u_{ti} = F_i(x_{ti}; \hat{\theta}_i)$.

- Semiparametric inference:
    - Estimate marginals nonparametrically via empirical CDFs $\hat{F}_i$.
    - Estimate copula parameters from pseudo-observations

        $$ \hat{u}_{ti} = \hat{F}_i(x_{ti}). $$

Important properties:
- Invariance: copulas are invariant under strictly increasing transformations of marginals.
- Tail dependence: measures the likelihood of joint extreme events.

For a bivariate copula $C$, the tail dependence coefficients are

$$ \lambda_U = \lim_{u \to 1^-} \frac{1 - 2u + C(u,u)}{1-u}, $$
$$ \lambda_L = \lim_{u \to 0^+} \frac{C(u,u)}{u}. $$

These coefficients capture asymptotic dependence in the upper and lower tails.

#### Common copula families
- Gaussian copula:

    $$ C_\Sigma(u,v) = \Phi_\Sigma(\Phi^{-1}(u), \Phi^{-1}(v)), $$

    where $\Phi_\Sigma$ is the multivariate normal CDF and $\Phi^{-1}$ the inverse standard normal CDF. It yields symmetric dependence with zero tail dependence in the bivariate case.

- Student-$t$ copula:

    $$ C_{\nu,\Sigma}(u,v) = t_{\nu,\Sigma}(t_\nu^{-1}(u), t_\nu^{-1}(v)), $$

    with degrees of freedom $\nu$. It accommodates heavier tails and nonzero tail dependence.

- Archimedean copulas:

    $$ C(u_1, \ldots, u_d) = \varphi^{-1}\big(\varphi(u_1) + \cdots + \varphi(u_d)\big), $$

    where $\varphi$ is a decreasing generator. Examples include Clayton, Gumbel, and Frank; these models capture asymmetric dependence and particular tail behaviors.

#### Financial applications
In finance, copulas separate marginal risk characteristics from joint dependence, making them suitable for:

- Credit risk: default probabilities of obligors are marginals, while joint default behavior is encoded by the copula.
- Portfolio risk: asset returns or losses have individual distributions, and the copula determines joint loss behavior.
- Structured products: valuation of CDO tranches depends on dependence among underlying credit events.

Example applications:
- Portfolio loss distribution:

    $$ L = \sum_{i=1}^d w_i \ell_i(X_i), $$

    where each loss component $\ell_i(X_i)$ has a marginal law and joint dependence given by a copula. The multivariate law is necessary to compute Value-at-Risk and Expected Shortfall.

- Default correlation modeling:
    - Use a latent variable representation with default thresholds as marginals and dependence encoded by a Gaussian or Student-$t$ copula.
    - The probability of $k$ defaults depends on the joint CDF of the latent factors.

#### Limitations and critical considerations
- Model risk: misspecified copulas can underestimate joint extremes.
- Calibration: inference requires sufficient data and robust estimation of marginals and dependence.
- Dynamic dependence: many copula models are static; financial dependence often evolves over time, motivating time-varying or regime-switching copulas.

This academic treatment emphasizes that copulas are fundamental tools for modeling multivariate distributions, allowing researchers to decouple marginals from dependence, estimate each component separately, quantify tail behavior, and apply these results to financial settings where joint extreme outcomes and credit dependence are central.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display

# -------------------------
# Parametri
# -------------------------
N = 3000

# Marginali
marginal1 = stats.norm(loc=0, scale=1)
marginal2 = stats.gamma(a=2.0, scale=1.0)

# Figura
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
plt.subplots_adjust(wspace=0.3, hspace=0.35)

# Seed fisso per avere confronti coerenti
base_rng = np.random.default_rng(12345)

# Campioni gaussiani standard indipendenti
z0 = base_rng.standard_normal((N, 2))

def generate_correlated_normals(rho):
    """
    Costruisce una coppia di gaussiane correlate
    partendo dagli stessi campioni base.
    """
    rho = np.clip(rho, -0.99, 0.99)

    x = z0[:, 0]
    y = rho * z0[:, 0] + np.sqrt(1 - rho**2) * z0[:, 1]

    return np.column_stack([x, y])

def update(rho):

    rho = float(rho)

    # Pulizia assi
    for ax in axes.ravel():
        ax.clear()

    # Gaussian copula
    z = generate_correlated_normals(rho)

    u = stats.norm.cdf(z[:, 0])
    v = stats.norm.cdf(z[:, 1])

    x = marginal1.ppf(u)
    y = marginal2.ppf(v)

    # -----------------
    # Marginale 1
    # -----------------
    ax = axes[0, 0]
    ax.hist(x, bins=40, color="C0", alpha=0.8)
    ax.set_title("Marginale 1 (Normale)")
    ax.set_xlabel("x")
    ax.set_ylabel("Freq")

    # -----------------
    # Copula
    # -----------------
    ax = axes[0, 1]
    ax.scatter(u, v, s=6, alpha=0.35, color="C3")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f"Copula gaussiana (ρ = {rho:.2f})")
    ax.set_xlabel("u")
    ax.set_ylabel("v")

    # -----------------
    # Congiunta
    # -----------------
    ax = axes[1, 0]
    hb = ax.hexbin(
        x,
        y,
        gridsize=40,
        cmap="Blues",
        mincnt=1
    )

    ax.set_title("Distribuzione congiunta")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    # -----------------
    # Marginale 2
    # -----------------
    ax = axes[1, 1]
    ax.hist(y, bins=40, color="C1", alpha=0.8)
    ax.set_title("Marginale 2 (Gamma)")
    ax.set_xlabel("y")
    ax.set_ylabel("Freq")

    fig.canvas.draw()

# Widget
slider = widgets.FloatSlider(
    value=0.5,
    min=-0.99,
    max=0.99,
    step=0.01,
    description="ρ",
    continuous_update=False
)

widgets.interact(update, rho=slider)

plt.show()